[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/assoudi-typica-ai/magbert-ner-ar-tutorial/blob/main/MagBERT_NER_AR_Tutorial.ipynb)

# MagBERT-NER-AR — Quick Tutorial

[**TypicaAI/MagBERT-NER-AR**](https://huggingface.co/TypicaAI/MagBERT-NER-AR) is an Arabic (MSA) named entity recognition model fine-tuned from `asafaya/bert-base-arabic` on a corpus curated from **Moroccan newspapers**. It works on standard Arabic news text and is especially well adapted to Moroccan entities: names, cities and provinces, institutions, laws and date conventions.

This notebook shows how to:
1. load the model,
2. extract entities from Arabic text,
3. visualize the results on a few Moroccan-context examples.

*Runs on CPU or GPU.*

## 1. Install and load

In [49]:
!pip install -q -U transformers spacy

In [50]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForTokenClassification

MODEL_ID = "TypicaAI/MagBERT-NER-AR"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForTokenClassification.from_pretrained(MODEL_ID).to(device).eval()

entity_types = sorted({l.split("-", 1)[1] for l in model.config.id2label.values() if "-" in l})
print("Device:", device)
print("Entity types:", entity_types)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Device: cpu
Entity types: ['CARDINAL', 'DATE', 'EVENT', 'GPE', 'LAW', 'LOC', 'NORP', 'ORDINAL', 'ORG', 'PERCENT', 'PERSON', 'PRODUCT']


## 2. Quick start with the Hugging Face pipeline

The simplest way to run the model:

In [51]:
from transformers import pipeline

ner = pipeline("token-classification", model=model, tokenizer=tokenizer,
               aggregation_strategy="first", device=0 if device == "cuda" else -1)

ner("يعقد بنك المغرب اجتماعه الفصلي بالرباط يوم الثلاثاء المقبل.")

[{'entity_group': 'ORG',
  'score': np.float32(0.8310782),
  'word': 'بنك',
  'start': 5,
  'end': 8},
 {'entity_group': 'ORG',
  'score': np.float32(0.6799539),
  'word': 'المغرب',
  'start': 9,
  'end': 15},
 {'entity_group': 'EVENT',
  'score': np.float32(0.44714493),
  'word': 'اجتماعه',
  'start': 16,
  'end': 23},
 {'entity_group': 'EVENT',
  'score': np.float32(0.475164),
  'word': 'الفصلي',
  'start': 24,
  'end': 30},
 {'entity_group': 'GPE',
  'score': np.float32(0.9324175),
  'word': 'بالرباط',
  'start': 31,
  'end': 38},
 {'entity_group': 'DATE',
  'score': np.float32(0.47412193),
  'word': 'يوم',
  'start': 39,
  'end': 42},
 {'entity_group': 'DATE',
  'score': np.float32(0.42294255),
  'word': 'الثلاثاء',
  'start': 43,
  'end': 51},
 {'entity_group': 'DATE',
  'score': np.float32(0.8938474),
  'word': 'المقبل',
  'start': 52,
  'end': 58}]

The model uses the **BILUO** tagging scheme (`B`egin, `I`nside, `L`ast, `U`nit, `O`utside). The generic pipeline only understands `B-`/`I-`, so multi-word entities can come back split into pieces.

For clean entity spans, use the helper below.

## 3. Recommended: BILUO-aware entity extraction

In [52]:
def extract_entities(text, max_length=512):
    """Return a list of entities: {type, text, score, start, end}."""
    enc = tokenizer(text, return_offsets_mapping=True, truncation=True,
                    max_length=max_length, return_tensors="pt")
    offsets = enc.pop("offset_mapping")[0].tolist()
    word_ids = enc.word_ids(0)
    with torch.no_grad():
        probs = model(**enc.to(device)).logits[0].softmax(-1).cpu()

    # 1) one label per word (first sub-token), span extended to the full word
    words, prev = [], None
    for i, wid in enumerate(word_ids):
        if wid is None:
            continue
        s, e = offsets[i]
        if wid != prev:
            score, idx = probs[i].max(-1)
            words.append([s, e, model.config.id2label[int(idx)], float(score)])
            prev = wid
        else:
            words[-1][1] = e

    # 2) group words into entities following BILUO
    entities, cur = [], None
    def close():
        nonlocal cur
        if cur:
            entities.append({"type": cur["type"], "text": text[cur["start"]:cur["end"]],
                             "score": round(sum(cur["scores"]) / len(cur["scores"]), 3),
                             "start": cur["start"], "end": cur["end"]})
            cur = None

    for s, e, tag, score in words:
        if tag in ("O", "PAD"):
            close(); continue
        prefix, _, etype = tag.partition("-")
        if prefix in ("B", "U") or cur is None or cur["type"] != etype:
            close()
            cur = {"type": etype, "start": s, "end": e, "scores": [score]}
        else:
            cur["end"] = e; cur["scores"].append(score)
        if prefix in ("L", "U"):
            close()
    close()
    return entities


pd.DataFrame(extract_entities("يعقد بنك المغرب اجتماعه الفصلي بالرباط يوم الثلاثاء المقبل."))

,type,text,score,start,end
0,ORG,بنك المغرب,0.756,5,15
1,EVENT,اجتماعه الفصلي,0.461,16,30
2,GPE,بالرباط,0.932,31,38
3,DATE,يوم,0.474,39,42
4,DATE,الثلاثاء المقبل,0.658,43,58


## 4. Visualization helper (right-to-left)

In [53]:
import spacy
from spacy import displacy
from IPython.display import display, HTML

_blank = spacy.blank("xx")

def show(text):
    ents = extract_entities(text)
    doc = _blank.make_doc(text)
    spans = [doc.char_span(e["start"], e["end"], label=e["type"], alignment_mode="expand") for e in ents]
    doc.ents = spacy.util.filter_spans([s for s in spans if s is not None])
    html = displacy.render(doc, style="ent", jupyter=False)
    display(HTML(f'<div dir="rtl" style="text-align:right;font-size:20px;line-height:2.3">{html}</div>'))
    if ents:
        display(pd.DataFrame(ents)[["type", "text", "score"]])

## 5. Examples — Moroccan news context

### Institutions & places

In [54]:
show("استقبلت جامعة الأخوين في إفران وفدا من جامعة محمد الخامس يوم الاثنين.")

,type,text,score
0,GPE,جامعة الأخوين,0.599
1,GPE,إفران,0.859
2,ORG,جامعة,0.315
3,GPE,محمد الخامس,0.463
4,DATE,الاثنين,0.628


### Statistics

In [55]:
show("أعلنت المندوبية السامية للتخطيط أن معدل البطالة بلغ حوالي 13 في المائة سنة 2024.")

,type,text,score
0,ORG,المندوبية السامية للتخطيط,0.791
1,DATE,سنة 2024,0.748


### Local administration & people

In [56]:
show("التقى رئيس جماعة شفشاون، كريم الإدريسي، بممثلي جمعيات المجتمع المدني.")

,type,text,score
0,GPE,شفشاون,0.808
1,PERSON,كريم الإدريسي,0.990


### Events, groups & Moroccan date format

In [57]:
show("يحتفل المغاربة بذكرى المسيرة الخضراء في 6 نونبر من كل سنة.")

,type,text,score
0,NORP,المغاربة,0.821
1,EVENT,المسيرة الخضراء,0.798
2,DATE,6 نونبر من,0.782
3,DATE,سنة,0.589


### Parliament & laws

In [58]:
show("صادق مجلس النواب على مشروع قانون يتعلق بمدونة الأسرة.")

,type,text,score
0,ORG,مجلس النواب,0.439


### Sport

In [59]:
show("حل فريق المغرب الفاسي في المرتبة الثالثة بعد الجولة العاشرة من البطولة الوطنية.")

,type,text,score
0,GPE,المغرب,0.389
1,ORG,الفاسي,0.567


### Regions & tourism

In [60]:
show("تعرف واحة سكورة في إقليم ورزازات إقبالا متزايدا من السياح الأوروبيين.")

,type,text,score
0,GPE,واحة سكورة,0.773
1,GPE,إقليم,0.629
2,GPE,ورزازات,0.789


### International

In [61]:
show("زار وفد من الاتحاد الإفريقي مدينة العيون للاطلاع على مشاريع التنمية.")

,type,text,score
0,ORG,الاتحاد الإفريقي,0.542
1,GPE,مدينة العيون,0.734


## 6. Beyond Morocco: other Arab-country conventions

The model handles standard Arabic from any country. Here is the same sentence written with Moroccan, Egyptian and Levantine month names:

In [62]:
examples = {
    "Morocco": "انعقد المؤتمر يوم 15 دجنبر في الدار البيضاء بحضور ممثلين عن وزارة الصحة.",
    "Egypt":   "انعقد المؤتمر يوم 15 ديسمبر في القاهرة بحضور ممثلين عن وزارة الصحة.",
    "Levant":  "انعقد المؤتمر يوم 15 كانون الأول في بيروت بحضور ممثلين عن وزارة الصحة.",
}

for region, sent in examples.items():
    print(f"── {region}")
    show(sent)

── Morocco


,type,text,score
0,DATE,15 دجنبر,0.940
1,GPE,الدار البيضاء,0.948
2,ORG,وزارة,0.402


── Egypt


,type,text,score
0,DATE,15 ديسمبر,0.882
1,GPE,القاهرة,0.942
2,ORG,وزارة الصحة,0.516


── Levant


,type,text,score
0,DATE,15 كانون الأول,0.882
1,GPE,بيروت,0.956
2,ORG,وزارة,0.451


## 7. Process several texts at once

In [63]:
texts = [sent for _, sent in examples.items()]

rows = [{"sentence": i, **e} for i, t in enumerate(texts) for e in extract_entities(t)]
results = pd.DataFrame(rows)

print(f"{len(results)} entities found in {len(texts)} sentences\n")
print(results["type"].value_counts().to_string())
results

9 entities found in 3 sentences

type
DATE    3
GPE     3
ORG     3


,sentence,type,text,score,start,end
0,0,DATE,15 دجنبر,0.940,18,26
1,0,GPE,الدار البيضاء,0.948,30,43
2,0,ORG,وزارة,0.402,60,65
3,1,DATE,15 ديسمبر,0.882,18,27
4,1,GPE,القاهرة,0.942,31,38
5,1,ORG,وزارة الصحة,0.516,55,66
6,2,DATE,15 كانون الأول,0.882,18,32
7,2,GPE,بيروت,0.956,36,41
8,2,ORG,وزارة,0.451,58,63


## 8. Try your own text

In [64]:
text = "تنظم وزارة الثقافة مهرجان فاس للموسيقى العريقة بمشاركة فنانين من السنغال ومالي."  #@param {type:"string"}
show(text)

,type,text,score
0,ORG,وزارة الثقافة,0.741
1,GPE,فاس,0.458
2,GPE,السنغال,0.705
3,GPE,ومالي,0.518


## Notes & limitations

* **Input:** Modern Standard Arabic in Arabic script. Darija, Arabizi and Latin-script text are out of scope.
* **Length:** the model was trained on sentences; split long documents into sentences before inference (max 512 sub-tokens).
* **Clitics:** prefixes such as *ب، ل، و* stay attached to the entity (e.g. *بالرباط*).
* **Rare types** (`LAW`, `ORDINAL`, `PRODUCT`, `LOC`) are less reliable than `PERSON`, `GPE`, `ORG`, `DATE`.
* **License:** see the [model card](https://huggingface.co/TypicaAI/MagBERT-NER-AR); released for education and research.

*Model by Hicham Assoudi — [Typica.ai](https://typica.ai). First trained April 2021, retrained August 2023.*